In [36]:
import shutil
from pathlib import Path

# -----------------------------
# AYARLAR
# -----------------------------
dataset_dir = Path("/kaggle/input/anipict-keciler-dataset/keciler_dataset")
output_dir = Path("/kaggle/working/all_data")  # Tüm veri buraya kopyalanacak
images_out = output_dir / "images"
labels_out = output_dir / "labels"

images_out.mkdir(parents=True, exist_ok=True)
labels_out.mkdir(parents=True, exist_ok=True)

# -----------------------------
# TÜM SPLITLERİ BİRLEŞTİR
# -----------------------------
for split in ["train", "valid", "test"]:
    images_dir = dataset_dir / split / "images"
    labels_dir = dataset_dir / split / "labels"
    
    for label_file in labels_dir.glob("*.txt"):
        # İlgili resim dosyası
        img_file_jpg = images_dir / f"{label_file.stem}.jpg"
        img_file_png = images_dir / f"{label_file.stem}.png"
        if img_file_jpg.exists():
            img_file = img_file_jpg
        elif img_file_png.exists():
            img_file = img_file_png
        else:
            continue  # Resim yoksa atla
        
        # Dosyaları kopyala
        shutil.copy(img_file, images_out / img_file.name)
        shutil.copy(label_file, labels_out / label_file.name)

print(f"✅ Tüm veriler birleştirildi. Toplam klasör: {output_dir}")


✅ Tüm veriler birleştirildi. Toplam klasör: /kaggle/working/all_data


In [74]:
from pathlib import Path

# -----------------------------
# AYARLAR
# -----------------------------
dataset_dir = Path("/kaggle/working/all_data")
classes = [
    "Ankara keçisi",
    "Halep keçisi",
    "Honamlı keçisi",
    "Kilis keçisi",
    "Kıl keçisi",
    "Malta keçisi",
    "Norduz keçisi",
    "Saanen keçisi",
    "Yaban Keçisi"
]

# -----------------------------
# SINIF SAYIMI
# -----------------------------
def count_classes(dataset_dir):
    counts = {cls: 0 for cls in classes}
    labels_dir = dataset_dir / "labels"
    
    for label_file in labels_dir.glob("*.txt"):
        with open(label_file) as f:
            lines = f.readlines()
            for line in lines:
                cls_idx = int(line.split()[0])
                counts[classes[cls_idx]] += 1
    return counts

total_counts = count_classes(dataset_dir)

# -----------------------------
# YAZDIR
# -----------------------------
print("📊 Toplam sınıf dağılımı (tüm veriler birleştirilmiş):")
for cls, c in total_counts.items():
    print(f"{cls}: {c}")


📊 Toplam sınıf dağılımı (tüm veriler birleştirilmiş):
Ankara keçisi: 1023
Halep keçisi: 765
Honamlı keçisi: 410
Kilis keçisi: 314
Kıl keçisi: 639
Malta keçisi: 162
Norduz keçisi: 78
Saanen keçisi: 887
Yaban Keçisi: 3301


In [80]:
import shutil
from pathlib import Path

folder = Path("/kaggle/working/augmented_dataset")

if folder.exists() and folder.is_dir():
    shutil.rmtree(folder)
    print(f"{folder} klasörü ve içeriği silindi.")
else:
    print(f"{folder} bulunamadı.")


/kaggle/working/augmented_dataset klasörü ve içeriği silindi.


In [81]:
import os
import cv2
import shutil
from collections import defaultdict
import albumentations as A
import math

# ============================
# 1️⃣ Dataset yolunu ayarla
# ============================
base_path = '/kaggle/working/all_data'
images_dir = os.path.join(base_path, 'images')
labels_dir = os.path.join(base_path, 'labels')
output_path = '/kaggle/working/augmented_dataset'

os.makedirs(os.path.join(output_path, 'images'), exist_ok=True)
os.makedirs(os.path.join(output_path, 'labels'), exist_ok=True)

# ============================
# 2️⃣ Sınıf isimleri ve augment ayarları
# ============================
target_count = 700  # her sınıf için hedef
class_names = [
    "Ankara keçisi","Halep keçisi","Honamlı keçisi","Kilis keçisi",
    "Kıl keçisi","Malta keçisi","Norduz keçisi","Saanen keçisi","Yaban Keçisi"
]
label_name_map = {i: name for i, name in enumerate(class_names)}

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=30, p=0.7),
    A.RandomBrightnessContrast(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.5)
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

# ============================
# 3️⃣ Her sınıfın örneklerini say
# ============================
id_count = defaultdict(list)

for f in os.listdir(labels_dir):
    file_path = os.path.join(labels_dir, f)
    with open(file_path, 'r') as file:
        lines = file.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        label_id = int(float(parts[0]))
        id_count[label_id].append(f)

# ============================
# 4️⃣ Kopyalama ve augment
# ============================
for label_id, files in id_count.items():
    class_name = label_name_map[label_id]
    for label_file in files:
        # Resim ve label path
        label_path = os.path.join(labels_dir, label_file)
        img_file = label_file.replace('.txt', '.jpg')
        img_path = os.path.join(images_dir, img_file)
        if not os.path.exists(img_path):
            img_file = label_file.replace('.txt', '.png')
            img_path = os.path.join(images_dir, img_file)
        if not os.path.exists(img_path):
            print("⚠️ Eksik eşleşme bulundu, atlandı:", label_file)
            continue

        # Orijinali kopyala
        shutil.copy2(img_path, os.path.join(output_path, 'images', img_file))
        shutil.copy2(label_path, os.path.join(output_path, 'labels', label_file))

    # Sadece 700 altındaki sınıfları augment et
    if len(files) < target_count and len(files) > 0:
        aug_factor = math.ceil((target_count - len(files)) / len(files))
        for label_file in files:
            label_path = os.path.join(labels_dir, label_file)
            img_file = label_file.replace('.txt', '.jpg')
            img_path = os.path.join(images_dir, img_file)
            if not os.path.exists(img_path):
                img_file = label_file.replace('.txt', '.png')
                img_path = os.path.join(images_dir, img_file)
            if not os.path.exists(img_path):
                continue

            with open(label_path, 'r') as f:
                lines = f.readlines()

            bboxes, labels_list = [], []
            image = cv2.imread(img_path)
            if image is None:
                print("⚠️ Görsel okunamadı, atlandı:", img_path)
                continue
            h_img, w_img = image.shape[:2]

            for line in lines:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                lbl_id = int(float(parts[0]))
                xc, yc, w, h = map(float, parts[1:])
                labels_list.append(lbl_id)
                x_min = (xc - w / 2) * w_img
                y_min = (yc - h / 2) * h_img
                x_max = (xc + w / 2) * w_img
                y_max = (yc + h / 2) * h_img
                bboxes.append([x_min, y_min, x_max, y_max])

            for i in range(aug_factor):
                augmented = transform(image=image, bboxes=bboxes, class_labels=labels_list)
                aug_img = augmented['image']
                aug_bboxes = augmented['bboxes']
                aug_labels = augmented['class_labels']

                new_img = img_file.replace('.jpg', f'_aug{i}.jpg').replace('.png', f'_aug{i}.png')
                cv2.imwrite(os.path.join(output_path, 'images', new_img), aug_img)

                new_label = new_img.replace('.jpg', '.txt').replace('.png', '.txt')
                label_lines = []
                for l, (x_min, y_min, x_max, y_max) in zip(aug_labels, aug_bboxes):
                    xc = ((x_min + x_max) / 2) / w_img
                    yc = ((y_min + y_max) / 2) / h_img
                    w  = (x_max - x_min) / w_img
                    h  = (y_max - y_min) / h_img
                    xc, yc, w, h = max(0, min(1, xc)), max(0, min(1, yc)), max(0, min(1, w)), max(0, min(1, h))
                    label_lines.append(f"{l} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
                with open(os.path.join(output_path, 'labels', new_label), 'w') as f:
                    f.writelines(label_lines)

print("✅ Augmentation tamamlandı. 700 altında olan tüm sınıflar hedef sayıya tamamlandı.")


✅ Augmentation tamamlandı. 700 altında olan tüm sınıflar hedef sayıya tamamlandı.


In [83]:
from pathlib import Path

# -----------------------------
# AYARLAR
# -----------------------------
dataset_dir = Path("/kaggle/working/augmented_dataset")
classes = [
    "Ankara keçisi",
    "Halep keçisi",
    "Honamlı keçisi",
    "Kilis keçisi",
    "Kıl keçisi",
    "Malta keçisi",
    "Norduz keçisi",
    "Saanen keçisi",
    "Yaban Keçisi"
]

# -----------------------------
# SINIF SAYIMI
# -----------------------------
def count_classes(dataset_dir):
    counts = {cls: 0 for cls in classes}
    labels_dir = dataset_dir / "labels"
    
    for label_file in labels_dir.glob("*.txt"):
        with open(label_file) as f:
            lines = f.readlines()
            for line in lines:
                cls_idx = int(float(line.split()[0]))  # YOLO label float -> int
                counts[classes[cls_idx]] += 1
    return counts

# -----------------------------
# ÇALIŞTIR
# -----------------------------
total_counts = count_classes(dataset_dir)

# -----------------------------
# YAZDIR
# -----------------------------
print("📊 Toplam sınıf dağılımı (tüm veriler birleştirilmiş):")
for cls, c in total_counts.items():
    print(f"{cls}: {c}")


📊 Toplam sınıf dağılımı (tüm veriler birleştirilmiş):
Ankara keçisi: 1023
Halep keçisi: 765
Honamlı keçisi: 814
Kilis keçisi: 940
Kıl keçisi: 1275
Malta keçisi: 810
Norduz keçisi: 702
Saanen keçisi: 887
Yaban Keçisi: 3301


In [85]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

# -----------------------------
# AYARLAR
# -----------------------------
dataset_dir = Path("/kaggle/working/augmented_dataset")
output_dir = Path("/kaggle/working/keciler_final_dataset")
train_ratio = 0.8
valid_ratio = 0.1
test_ratio  = 0.1

# -----------------------------
# ÇIKTI KLASÖRLERİ
# -----------------------------
for split in ["train", "valid", "test"]:
    (output_dir / split / "images").mkdir(parents=True, exist_ok=True)
    (output_dir / split / "labels").mkdir(parents=True, exist_ok=True)

# -----------------------------
# Tüm resim ve label dosyalarını eşleştir
# -----------------------------
images = list((dataset_dir / "images").glob("*.*"))
labels = list((dataset_dir / "labels").glob("*.txt"))

# Label ile image eşleşmesini yap
image_map = {img.stem: img for img in images}
label_map = {lbl.stem: lbl for lbl in labels}

common_keys = list(set(image_map.keys()) & set(label_map.keys()))
common_keys.sort()

# -----------------------------
# Split için indeks oluştur
# -----------------------------
train_keys, temp_keys = train_test_split(common_keys, train_size=train_ratio, random_state=42)
valid_keys, test_keys = train_test_split(temp_keys, test_size=test_ratio/(test_ratio+valid_ratio), random_state=42)

splits = {
    "train": train_keys,
    "valid": valid_keys,
    "test" : test_keys
}

# -----------------------------
# Dosyaları kopyala
# -----------------------------
for split, keys in splits.items():
    for key in keys:
        # Image kopyala
        shutil.copy2(image_map[key], output_dir / split / "images" / image_map[key].name)
        # Label kopyala
        shutil.copy2(label_map[key], output_dir / split / "labels" / label_map[key].name)

print(f"✅ Dataset başarıyla split edildi ve '{output_dir}' içerisine kopyalandı.")
print("Train / Valid / Test oranları:", train_ratio, valid_ratio, test_ratio)


✅ Dataset başarıyla split edildi ve '/kaggle/working/keciler_final_dataset' içerisine kopyalandı.
Train / Valid / Test oranları: 0.8 0.1 0.1


In [88]:
from pathlib import Path
from collections import defaultdict

# -----------------------------
# AYARLAR
# -----------------------------
dataset_dir = Path("/kaggle/working/keciler_final_dataset")
classes = [
    "Ankara keçisi",
    "Halep keçisi",
    "Honamlı keçisi",
    "Kilis keçisi",
    "Kıl keçisi",
    "Malta keçisi",
    "Norduz keçisi",
    "Saanen keçisi",
    "Yaban Keçisi"
]

# -----------------------------
# Fonksiyon: split için sınıf sayımı
# -----------------------------
def count_classes_split(split_dir):
    counts = defaultdict(int)
    labels_dir = split_dir / "labels"
    for label_file in labels_dir.glob("*.txt"):
        with open(label_file) as f:
            lines = f.readlines()
            for line in lines:
                cls_idx = int(float(line.split()[0]))
                counts[classes[cls_idx]] += 1
    return counts

# -----------------------------
# Her split'i say
# -----------------------------
splits = ["train", "valid", "test"]
split_counts = {split: count_classes_split(dataset_dir / split) for split in splits}

# -----------------------------
# Toplamları hesapla ve yazdır
# -----------------------------
print("📊 Sınıf dağılımı (train / valid / test / toplam):")
for i, cls in enumerate(classes):
    train_c = split_counts["train"].get(cls, 0)
    valid_c = split_counts["valid"].get(cls, 0)
    test_c  = split_counts["test"].get(cls, 0)
    total_c = train_c + valid_c + test_c
    print(f"{cls}: train={train_c}, valid={valid_c}, test={test_c}, toplam={total_c}")


📊 Sınıf dağılımı (train / valid / test / toplam):
Ankara keçisi: train=669, valid=179, test=175, toplam=1023
Halep keçisi: train=588, valid=60, test=117, toplam=765
Honamlı keçisi: train=634, valid=82, test=98, toplam=814
Kilis keçisi: train=790, valid=67, test=83, toplam=940
Kıl keçisi: train=1100, valid=67, test=108, toplam=1275
Malta keçisi: train=623, valid=100, test=87, toplam=810
Norduz keçisi: train=569, valid=72, test=61, toplam=702
Saanen keçisi: train=700, valid=81, test=106, toplam=887
Yaban Keçisi: train=2755, valid=299, test=247, toplam=3301


In [86]:
import shutil
from pathlib import Path

# -----------------------------
# Klasör ve zip yolu
# -----------------------------
dataset_dir = Path("/kaggle/working/keciler_final_dataset")
zip_path = Path("/kaggle/working/keciler_final_dataset.zip")

# -----------------------------
# Zip oluştur
# -----------------------------
shutil.make_archive(str(zip_path).replace('.zip',''), 'zip', root_dir=dataset_dir)

print(f"✅ Zip oluşturuldu: {zip_path}")


✅ Zip oluşturuldu: /kaggle/working/keciler_final_dataset.zip
